# Demo: Run a Classifier Directly on Images

### Objective
Infer and evaluate lesion classes (tumour / cyst) on KITS scans

### Prerequisites:

- `dataset.csv` with image and segmentation paths to KITS scans (see [Tuorital.ipynb](./tutorial.ipynb))
- RenalVision

## 1. Initialise your prediction model
If you followed the tutorial you can use your own model.pkl as an identifier, otherwise, chose our pretrained bundle.
(Our bundle was trained with the KITS data and will we positively biased.)

### Train

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

from renal_vision.modeling.inference import LesionPredictor

# 1. Load Data
data = pd.read_csv("../data/KITS.csv")
data = data.head(n=5) # [Optional] reduce files to save time


# 2. Initialize Predictor (Auto-loads extractor config from the model)
# predictor = LesionPredictor("model/model.pkl")
predictor = LesionPredictor("RADIOMICS_BINARY")

# 3. Define Labelmap
labelmap = {
    1:-1, # exclude kidney
    2:0, # First class: tumours
    3:1, # Second class: cysts
}

# 4. Iterate over Images
y_true = []
y_prob = []
for _, row in tqdm(data.iterrows(), total=len(data)):

    # filter for components of size >= 400m^3
    seg_filtered, metadata_list = predictor.filter_components(row["seg_path"])
    
    for lesion_id, metadata in enumerate(metadata_list, start=1):
        
        # skip ignored classes (kidney)
        class_id = labelmap[metadata["class_id"]]
        if class_id == -1:
            continue

        # Run inference
        prediction = predictor.infer_lesion(row["image_path"],seg_filtered==lesion_id)
        tqdm.write(f"Groundtruth: {class_id} \t Prediction: {prediction}")

        y_true.append(class_id)
        y_prob.append(prediction["probability"])

### Evaluate

In [ ]:
from renal_vision.shared.metrics import ModelEvaluator

class_names = ["Tumour", "Cyst"]
evaluator = ModelEvaluator(y_true, y_prob, class_names)

results = evaluator.get_scalars()
auc = evaluator.get_auc(with_ci=False)
ap = evaluator.get_ap(with_ci=False)
print(results)
print("AUC:", auc)
print("AP:", ap)

evaluator.plot_cm(figsize=(6,5))
evaluator.plot_roc(figsize=(6,5))
evaluator.plot_pr(figsize=(6,5))